In [18]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

**Запускаємо віртуальний дисплей**

In [19]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [20]:
import gymnasium as gym

env = gym.make("LunarLander-v2")

observation, info = env.reset()

for _ in range(20):
  action = env.action_space.sample()
  print("Action taken:", action)

  observation, reward, terminated, truncated, info = env.step(action)

  if terminated or truncated:
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 3
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 0
Action taken: 1
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 0
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 1
Action taken: 1
Action taken: 0


**Створюємо encironment, робимо випадкову дію, отримаємо статус та нагороду в залежності, якщо сплинув час або завдання провалено(розбиття в цьому випадку), то робимо reset.**

In [42]:
env = gym.make("LunarLander-v2")
env.reset()
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())

Observation Space Shape (8,)
Sample observation [-1.8583317e+01 -5.9806458e+01  1.6942939e+00 -4.0434093e+00
 -2.8192444e+00 -4.0423551e+00  2.1432271e-02  9.0236264e-01]


**Бачимо, що є 6 фіч у даної моделі**

In [29]:
env = make_vec_env('LunarLander-v2', n_envs=16)

**Створюємо векторизований environment**

In [30]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cpu device


**Створюємо модель**

In [31]:
from wasabi import Printer
import numpy as np
from stable_baselines3.common.base_class import BaseAlgorithm
from pathlib import Path
import tempfile
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    VecEnv,
    VecVideoRecorder,
)

In [32]:
msg = Printer()

In [33]:
def generate_replay(
    model: BaseAlgorithm,
    eval_env: VecEnv,
    video_length: int,
    is_deterministic: bool,
    local_path: Path,
):
    with tempfile.TemporaryDirectory() as tmpdirname:
        env = VecVideoRecorder(
            eval_env,
            tmpdirname,
            record_video_trigger=lambda x: x == 0,
            video_length=video_length,
            name_prefix="",
        )

        obs = env.reset()
        lstm_states = None
        episode_starts = np.ones((env.num_envs,), dtype=bool)

        try:
            for _ in range(video_length):
                action, lstm_states = model.predict(
                    obs,
                    state=lstm_states,
                    episode_start=episode_starts,
                    deterministic=is_deterministic,
                )
                obs, _, episode_starts, _ = env.step(action)

            env.close()

            inp = env.video_recorder.path
            out = local_path
            os.system(f"ffmpeg -y -i {inp} -vcodec h264 {out}".format(inp, out))
            print(f"Video saved to: {out}")
        except KeyboardInterrupt:
            pass
        except Exception as e:
            msg.fail(str(e))
            msg.fail(
                "We are unable to generate a replay of your agent"
            )

**Робимо імпорти та створюємо функцію для запису реплею**

In [34]:
model.learn(total_timesteps=1000000)
model_name = "ppo-LunarLander-v2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.4     |
|    ep_rew_mean     | -179     |
| time/              |          |
|    fps             | 4754     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90.8        |
|    ep_rew_mean          | -150        |
| time/                   |             |
|    fps                  | 2911        |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.005591118 |
|    clip_fraction        | 0.0411      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.0015      |
|    learning_rate        | 0.

**Тренуємо PPO agent**

In [40]:
import os
video_dir = "."
video_name = "replay.mp4"
env_id = "LunarLander-v2"

generate_replay(
    model = model,
    eval_env=DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))]),
    video_length=3000,
    is_deterministic=True,
    local_path=os.path.join(video_dir, video_name)
)

Saving video to /tmp/tmpxpxamfh9/-step-0-to-step-3000.mp4
Moviepy - Building video /tmp/tmpxpxamfh9/-step-0-to-step-3000.mp4.
Moviepy - Writing video /tmp/tmpxpxamfh9/-step-0-to-step-3000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmpxpxamfh9/-step-0-to-step-3000.mp4
Video saved to: ./replay.mp4


In [41]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=600 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

**Візуалізуємо тренування**

In [37]:
eval_env = Monitor(gym.make("LunarLander-v2", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=257.37 +/- 22.664624207293603


**Оцінюємо модель, вийшло більше 200, що означає модель добре натренована за дану кількість повторювань. Це також підтверджує отримане відео**